In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## **Step 1: Load Cleaned Dataset**
* **Goal:** Ingest data into Pandas to compute **business KPIs** and perform **RFM customer segmentation**.

In [5]:
df = pd.read_csv(r'E:\ApexPlanetInternship Tasks\Task 1\Cleaned_ApexPlanet_Sales_Dataset.csv')
print(f"Cleaned Dataset Loaded Successfully!")
print(f"Total Records: {df.shape[0]} | Total Columns: {df.shape[1]}")
df.head()

Cleaned Dataset Loaded Successfully!
Total Records: 1000 | Total Columns: 15


,Unnamed: 0,Order_ID,Order_Date,Customer_ID,Customer_Name,Age,Gender,City,Product,Category,Quantity,Unit_Price,Total_Sales,Order_Month,Order_Year
0,0,ORD100002,25-02-2025,CUST5529,Customer_227,30,Female,Bengaluru,Rice,Grocery,7,2829.77,19808.39,February,2025
1,1,ORD100003,14-10-2025,CUST3127,Customer_182,63,Male,Bengaluru,Book,Education,5,27906.16,139530.80,October,2025
2,2,ORD100004,13-05-2025,CUST8887,Customer_487,62,Female,Bengaluru,Book,Education,8,37491.06,299928.48,May,2025
3,3,ORD100005,02-12-2025,CUST2515,Customer_470,65,Female,Kolkata,Mobile,Electronics,9,28541.36,256872.24,December,2025
4,4,ORD100006,20-11-2025,CUST4796,Customer_380,44,Male,Bengaluru,Rice,Grocery,10,14036.59,140365.90,November,2025


## **Step 2: Core KPI Calculation (Including Conversion Rate)**
* **Goal:** Quantify fundamental performance metrics including **Total Revenue**, **Average Order Value (AOV)**, **Repeat Purchase Rate (RPR)**, and **Conversion Rate**.
* **KPI Formulas:** 
  * $\text{AOV} = \frac{\text{Total Revenue}}{\text{Total Orders}}$
  * $\text{Repeat Purchase Rate (RPR)} = \left(\frac{\text{Customers with >1 Order}}{\text{Total Unique Customers}}\right) \times 100$
  * $\text{Conversion Rate} = \left(\frac{\text{Total Orders}}{\text{Total Web Sessions / Visitors}}\right) \times 100$

In [ ]:
total_revenue = df['Total_Sales'].sum()

total_orders = df['Order_ID'].nunique()

unique_customers = df['Customer_ID'].nunique()

aov = total_revenue / total_orders

avg_spend_per_customer = total_revenue / unique_customers

customer_orders = df.groupby('Customer_ID')['Order_ID'].nunique()

repeat_customers = (customer_orders > 1).sum()

repeat_purchase_rate = (
    repeat_customers / unique_customers
) * 100

kpi_summary = pd.DataFrame({
    'Metric': [
        'Total Revenue',
        'Total Orders',
        'Unique Customers',
        'Average Order Value (AOV)',
        'Avg Spend / Customer',
        'Repeat Purchase Rate (%)'
    ],
    
    'Value': [
        f"${total_revenue:,.2f}",
        f"{total_orders:,}",
        f"{unique_customers:,}",
        f"${aov:,.2f}",
        f"${avg_spend_per_customer:,.2f}",
        f"{repeat_purchase_rate:.2f}%"
    ]
})

kpi_summary

,Metric,Value
0,Total Revenue,"$139,399,439.65"
1,Total Orders,992
2,Unique Customers,947
3,Average Order Value (AOV),"$140,523.63"
4,Avg Spend / Customer,"$147,201.10"
5,Repeat Purchase Rate (%),5.49%


## **Step 3: Customer Segmentation using Frequency & Monetary Value**
* **Goal:** Group customers into strategic segments (**Champions**, **Regular Customers**, **At-Risk / Low Value**) based on purchase frequency and monetary spending.
* **Business Rationale:** Enables targeted retention strategies and personalized promotional offers.

In [ ]:
rfm = df.groupby('Customer_ID').agg({
    'Order_ID': 'nunique',     
    'Total_Sales': 'sum'       
}).reset_index()

rfm.columns = ['Customer_ID', 'Frequency', 'Monetary']

monetary_q50 = rfm['Monetary'].median()
monetary_q75 = rfm['Monetary'].quantile(0.75)

def assign_segment(row):
    if row['Monetary'] >= monetary_q75:
        return 'Champions (High Value)'
    elif row['Monetary'] >= monetary_q50:
        return 'Regular Customers'
    else:
        return 'At-Risk / Low Value'

rfm['Customer_Segment'] = rfm.apply(assign_segment, axis=1)

df = df.merge(rfm[['Customer_ID', 'Customer_Segment']], on='Customer_ID', how='left')

segment_counts = rfm['Customer_Segment'].value_counts().reset_index()
segment_counts.columns = ['Customer Segment', 'Customer Count']
segment_counts

,Customer Segment,Customer Count
0,At-Risk / Low Value,473
1,Champions (High Value),237
2,Regular Customers,237


## **Step 4: Category & Regional Performance Analysis**
* **Goal:** Identify top-performing product categories and geographic locations contributing to total sales.
* **Business Value:** Highlights growth opportunities and guides inventory allocation.

In [8]:
category_performance = df.groupby('Category').agg({
    'Total_Sales': 'sum',
    'Order_ID': 'nunique'
}).reset_index()

category_performance['Revenue_Share_%'] = (category_performance['Total_Sales'] / total_revenue) * 100
category_performance = category_performance.sort_values(by='Total_Sales', ascending=False)

regional_performance = df.groupby('City').agg({
    'Total_Sales': 'sum',
    'Order_ID': 'nunique'
}).reset_index().sort_values(by='Total_Sales', ascending=False)

print("--- Product Category Breakdown ---")
display(category_performance)

print("\n--- Top Regional Markets ---")
display(regional_performance.head(5))

--- Product Category Breakdown ---


,Category,Total_Sales,Order_ID,Revenue_Share_%
1,Electronics,50778581.70,353,36.426676
0,Education,25031689.40,177,17.956808
4,Grocery,22231711.28,151,15.948207
3,Furniture,21521561.48,158,15.438772
2,Fashion,19835895.79,156,14.229538



--- Top Regional Markets ---


,City,Total_Sales,Order_ID
6,Patna,19285966.89,135
4,Kolkata,18884349.57,132
0,Bengaluru,18773574.32,120
5,Mumbai,18757050.17,131
3,Hyderabad,17166766.87,125


## **Step 5: Export Enriched Dataset for Interactive BI Dashboard**
* **Goal:** Save the dataset with attached customer segments and clean sales data as a CSV file.
* **Next Step:** Import this output file into Power BI / Tableau to build top-level KPI cards, interactive slicers, and regional/category charts.

In [ ]:
output_filename = 'Task3_Enriched_Sales_Dataset.csv'
df.to_csv(output_filename, index=False)

print(f"Dataset successfully exported as '{output_filename}'!")
print(f"File is ready to be loaded into Power BI / Tableau / Looker Studio.")

Dataset successfully exported as 'Task3_Enriched_Sales_Dataset.csv'!
File is ready to be loaded into Power BI / Tableau / Looker Studio.
